## **TP3** *-Zakaria Rabbah-*

**0 - Imports**

In [32]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    precision_score,
    recall_score,
    f1_score
)

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

np.random.seed(42)
tf.random.set_seed(42)

**Q1) Chargement des données**

In [33]:
df = pd.read_csv("dataset_fraude.csv")
print("dimensions du dataset ",df.shape)
df.head()


dimensions du dataset  (5228, 11)


,amount,hour,merchant_category,country,country_risk,distance_km,tx_last_1h,tx_last_24h,device_change,is_international,label_fraud
0,11.95,12,online_services,MA,0.044,13.91,4,8,0,0,1
1,10.29,11,utilities,MA,0.136,13.17,1,4,0,0,0
2,97.04,19,pharmacy,MA,0.075,51.35,0,2,0,0,0
3,16.60,19,atm,MA,0.047,21.52,3,7,0,0,0
4,70.93,7,fuel,CN,0.396,47.09,0,2,0,1,0


**Q2) Analyse de la variable cible**

In [34]:
print("Nombre de transactions normales et frauduleuses: ")
print(df["label_fraud"].value_counts())

Nombre de transactions normales et frauduleuses: 
label_fraud
0    4451
1     777
Name: count, dtype: int64


**Q3) Séparation des variables**

In [35]:
X = df.drop("label_fraud", axis=1)
y = df["label_fraud"]

print("Dimensions de X :", X.shape)
print("Dimensions de y :", y.shape)

Dimensions de X : (5228, 10)
Dimensions de y : (5228,)


**Q4) Encodage des variables catégorielles**

In [36]:
# one hot encoding
X = pd.get_dummies(X, columns=["merchant_category", "country"], drop_first=False)

cols_bool = X.select_dtypes(include="bool").columns
X[cols_bool] = X[cols_bool].astype(int)

print("Nv dimension de X après encodage :", X.shape)
X.head()

Nv dimension de X après encodage : (5228, 30)


,amount,hour,country_risk,distance_km,tx_last_1h,tx_last_24h,device_change,is_international,merchant_category_atm,merchant_category_electronics,...,country_AE,country_BR,country_CN,country_ES,country_FR,country_MA,country_NG,country_RU,country_UK,country_US
0,11.95,12,0.044,13.91,4,8,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0
1,10.29,11,0.136,13.17,1,4,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0
2,97.04,19,0.075,51.35,0,2,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0
3,16.60,19,0.047,21.52,3,7,0,0,1,0,...,0,0,0,0,0,1,0,0,0,0
4,70.93,7,0.396,47.09,0,2,0,1,0,0,...,0,0,1,0,0,0,0,0,0,0


**Q5) Split des données**

In [37]:
# Step 1 : 70% train et 30% pour le reste
X_train, X_reste, y_train, y_reste = train_test_split(
    X, y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

# Step 2 : les 30% restants, 1/2 validation, 1/2 test
X_val, X_test, y_val, y_test = train_test_split(
    X_reste, y_reste,
    test_size=0.50,
    random_state=42,
    stratify=y_reste
)


print("X_train :", X_train.shape)
print("X_val   :", X_val.shape)
print("X_test  :", X_test.shape)

X_train : (3659, 30)
X_val   : (784, 30)
X_test  : (785, 30)


**Q6) Mise à echelle**

In [38]:
scaler = StandardScaler()

X_train_sc = scaler.fit_transform(X_train)
X_val_sc = scaler.transform(X_val)
X_test_sc = scaler.transform(X_test)

**Q7) Construction du modèle**

In [39]:
# Q7) Construction du modèle
model = keras.Sequential([
    layers.Input(shape=(X_train_sc.shape[1],)),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.30),
    layers.Dense(32, activation='relu'),
    layers.Dropout(0.20),
    layers.Dense(1, activation='sigmoid')
])

model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_3 (Dense)                 │ (None, 64)             │         1,984 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,097 (16.00 KB)

 Trainable params: 4,097 (16.00 KB)

 Non-trainable params: 0 (0.00 B)

**Q8) Compilation du modèle**

In [40]:
model.compile(
    loss='binary_crossentropy',
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    metrics=['accuracy', keras.metrics.Recall(name='recall')]
)

**Pourquoi le recall est-il une métrique importante dans la détection de fraude ?**

**Rep** : Le recall est important en détection de fraude car il mesure la capacité de modèle à détecter les transactions fraudes réelles. Une fraude non détectée FN peut avoir un coût financier important. Le recall doit etre élevé.

**Q9) Gestion du surapprentissage**

In [41]:
early_stop = keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

**Q10) Gestion du déséquilibre des classes**

In [42]:
neg = int((y_train == 0).sum())
pos = int((y_train == 1).sum())
class_weight = {0: 1.0, 1: (neg / max(pos, 1))}
print("\nclass_weight:", class_weight)


class_weight: {0: 1.0, 1: 5.726102941176471}


**Q11) Entraînement du modèle**

In [43]:
history = model.fit(
    X_train_sc, y_train,
    validation_data=(X_val_sc, y_val),
    epochs=200,
    batch_size=64,
    class_weight=class_weight,
    callbacks=[early_stop],
    verbose=1
)

Epoch 1/200
58/58 ━━━━━━━━━━━━━━━━━━━━ 3s 10ms/step - accuracy: 0.5132 - loss: 1.2154 - recall: 0.5298 - val_accuracy: 0.5536 - val_loss: 0.6890 - val_recall: 0.6724
Epoch 2/200
58/58 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.5960 - loss: 1.1292 - recall: 0.5934 - val_accuracy: 0.6097 - val_loss: 0.6508 - val_recall: 0.6638
Epoch 3/200
58/58 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.6295 - loss: 1.0891 - recall: 0.6309 - val_accuracy: 0.6059 - val_loss: 0.6431 - val_recall: 0.6379
Epoch 4/200
58/58 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6568 - loss: 1.0615 - recall: 0.6337 - val_accuracy: 0.6237 - val_loss: 0.6295 - val_recall: 0.6638
Epoch 5/200
58/58 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6717 - loss: 1.0436 - recall: 0.6637 - val_accuracy: 0.6161 - val_loss: 0.6359 - val_recall: 0.6466
Epoch 6/200
58/58 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.6559 - loss: 1.0394 - recall: 0.6790 - val_accuracy: 0.6403 - val_loss: 0.6194 - val_recall: 0.6552
Epoch 7/2

**Q13) Évaluation du modèle**

In [44]:
# Q13) Évaluation sur TEST

# Distribution des classes dans le test
print("\n--- Distribution des classes dans TEST ---")
print("Normal (0) :", (y_test == 0).sum())
print("Fraude (1) :", (y_test == 1).sum())

test_loss, test_acc, test_recall = model.evaluate(X_test_sc, y_test, verbose=0)
print(f"\nTEST -> loss={test_loss:.4f} | acc={test_acc:.4f} | recall={test_recall:.4f}")

# Probabilités + prédictions
y_proba = model.predict(X_test_sc).ravel()
y_pred = (y_proba >= 0.5).astype(int)

# Matrice de confusion
print("\n--- Confusion Matrix (seuil=0.5) ---")
cm = confusion_matrix(y_test, y_pred, labels= [1,0])
print(cm)

# Métriques
print("\n--- Metrics ---")
print("Precision :", precision_score(y_test, y_pred, zero_division=0))
print("Recall    :", recall_score(y_test, y_pred, zero_division=0))
print("F1        :", f1_score(y_test, y_pred, zero_division=0))


--- Distribution des classes dans TEST ---
Normal (0) : 668
Fraude (1) : 117

TEST -> loss=0.5873 | acc=0.6917 | recall=0.6496
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step

--- Confusion Matrix (seuil=0.5) ---
[[ 76  41]
 [201 467]]

--- Metrics ---
Precision : 0.2743682310469314
Recall    : 0.6495726495726496
F1        : 0.38578680203045684


**Combien de fausses alertes le modèle a-t-il générées ?**

Les fausses alertes = FP = 201

**Combien de fraudes le modèle n’a-t-il pas détectées ?**

Les fraudes non détectées = FN = 41

**Q14) Tester les métriques d’évaluation pour différents seuils de décision**

In [45]:
# Q14) Tester d'autres seuils
def evaluate_threshold(th):
    yp = (y_proba >= th).astype(int)
    p = precision_score(y_test, yp, zero_division=0)
    r = recall_score(y_test, yp, zero_division=0)
    f = f1_score(y_test, yp, zero_division=0)
    return p, r, f

print("\n--- Seuils alternatifs ---")
for th in [0.6, 0.5, 0.3, 0.2, 0.1]:
    p, r, f = evaluate_threshold(th)
    print(f"Seuil={th:.1f} -> Precision={p:.4f} | Recall={r:.4f} | F1={f:.4f}")


--- Seuils alternatifs ---
Seuil=0.6 -> Precision=0.3396 | Recall=0.4615 | F1=0.3913
Seuil=0.5 -> Precision=0.2744 | Recall=0.6496 | F1=0.3858
Seuil=0.3 -> Precision=0.1853 | Recall=0.8632 | F1=0.3051
Seuil=0.2 -> Precision=0.1631 | Recall=0.9231 | F1=0.2773
Seuil=0.1 -> Precision=0.1505 | Recall=0.9915 | F1=0.2613


**Quel seuil vous semble le plus adapté pour un système réel de détection de fraude ?**

Le seuil 0.3 semble offrir un bon compromis dans notre cas. Avec ce seuil, le recall atteint 86.32 %, ce qui signifie que la majorité des fraudes sont détectées. Bien que la précision soit plus faible, ce compromis reste acceptable dans un système de détection de fraude où la priorité est d'éviter les fraudes non détectées.